In [1]:
from __future__ import annotations
from typing import List, Dict, Tuple
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from pandas import DataFrame
import json
from collections import defaultdict
from itertools import product
from pprint import pprint
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import argparse

hytraits_path = (Path.cwd().parent/'hytraits').resolve()
if str(hytraits_path) not in sys.path:
    sys.path.append(str(hytraits_path))
import hytraits as H 
from utils import (get_paths,
                   get_traits)


def get_deploy_by_trait() -> Dict:
    '''
    Return: Dict
            Key: str; trait name
            Value: Dict; {str (treatment):, Path (deploy dir)}
    '''
    PATHS = get_paths()
    PATHS['post'].mkdir(parents=True, exist_ok=True)

    deploy_dirs = [e for e in PATHS['deploy'].glob('*') if e.is_dir()]
    deploy_dirs.sort()

    out = defaultdict(dict)
    for deploy_dir in deploy_dirs:
        model, treatment, train_on, split, test_on = deploy_dir.stem.split('__')
        out[train_on][treatment] = deploy_dir
        
    return out


def make_truepreds_for_trait(trait: str,
                             title: str,
                             color: str = '#EF476F') -> None:
    PATHS = get_paths()
    trait_deploys = get_deploy_by_trait()
    deploy_dirs = sorted(list(trait_deploys[trait].values()))

    agu_dir = Path('io/agu/truepreds') 
    agu_dir.mkdir(parents=True, exist_ok=True)
    
    for deploy_dir in deploy_dirs:
        tkns = deploy_dir.stem.split('__')
        treatment, trait = tkns[1], tkns[2]

        preds_df = pd.read_csv(deploy_dir/'preds.csv')

        colors_df = pd.DataFrame({'sample_id': preds_df['sample_id']})
        colors_df['color'] = color

        metrics_df = pd.read_csv(deploy_dir/'metrics.csv')
        r2 = np.mean(metrics_df['r2'].values)
        rnrmse = np.mean(metrics_df['range_normalized_rmse'].values)
        stats = (f'R2 = {r2:.2f}\n'
                 f'RNRMSE = {rnrmse:.1f}')

        # with sdevs
        fig, ax = plt.subplots(figsize=(12, 12))
        ax = H.plot_pred_vs_true(ax,
                                 pred_df=preds_df,
                                 color_df=colors_df)
        ax.text(0.07, 
                0.85, 
                stats, 
                fontsize=9,
                transform=ax.transAxes,
                horizontalalignment='left')
        fig.suptitle(title, y=0.99, fontsize=24)
        plt.tight_layout()
        plt.savefig(agu_dir/f'{trait}_{treatment}_truepreds_with_sdev.jpg')
        plt.close()

        # without sdevs
        fig, ax = plt.subplots(figsize=(12, 12))
        ax = H.plot_pred_vs_true(ax,
                                 pred_df=preds_df,
                                 color_df=colors_df,
                                 show_sdev=False)
        ax.text(0.07, 
                0.85, 
                stats, 
                fontsize=16,
                transform=ax.transAxes,
                horizontalalignment='left')
        fig.suptitle(title, y=0.99, fontsize=24)
        plt.tight_layout()
        plt.savefig(agu_dir/f'{trait}_{treatment}_truepreds_without_sdev.jpg')
        plt.close()


def get_eda_by_trait() -> Dict:
    '''
    EDA directories per trait.

    Return: Dict; {str (trait): List[Tuple[str, Path]] ([(treatment, eda dir)])}
    '''
    PATHS = get_paths()
    TRAITS = get_traits()
    
    suffixes = ['asis-raw', 'asis-log', 'move-raw', 'move-log', 'move-pa-raw', 'move-pa-log']
    d = defaultdict(list)
    for trait in TRAITS.values():
        for suffix in suffixes:
            d[trait].append((suffix, PATHS['eda']/f'{trait}-{suffix}-eda'))
    return d
    
def make_ndi_r2_histogram_for_trait(trait: str,
                                    title: str,
                                    color: str = '#EF476F') -> None:
    PATHS = get_paths()
    trait_edas = get_eda_by_trait()
    eda_dirs = [d for (_, d) in trait_edas[trait]]

    agu_dir = Path('io/agu/ndir2hists') 
    agu_dir.mkdir(parents=True, exist_ok=True)

            
    for eda_dir in eda_dirs:
        # ndi R2 histogram
        data = np.load(eda_dir/'ndi-r2.npz')
        metrics, wavelengths = data['metrics'], data['wavelengths']
        cmap = plt.cm.viridis
        cmap.set_under('white')
        norm = Normalize(vmin=0, vmax=1.0)
        fig, ax = plt.subplots(figsize=(12, 12))
        ax = H.plot_ndi_as_histogram(ax=ax,
                                     metrics=metrics,
                                     min_max=(0, 1),
                                     color=color)
        ax.tick_params(axis='x', labelrotation=90)

        fig.suptitle(title, y=0.99, fontsize=24)
        plt.tight_layout()
        plt.savefig(agu_dir/f'{eda_dir.stem}_ndir2hist.jpg')
        plt.close()

In [2]:
COLORS = ['#EF476F', '#F78C6B', '#FFD166', '#06D6A0', '#118A32', '#073B4C']
TITLES = {'cfchl': 'Chlorophyll (RFU)',
          'cfpc': 'Phycocyanin (RFU)',
          'tss': 'Total Suspended Solids (mg/L)',
          'npoc': 'Dissolved Organic Carbon (mg/L)',
          'no23': 'nitrate + nitrite as N (µg/L)',
          'secchi': 'Secchi Depth (m)',
          'tn': 'Total Nitrogen (µg/L)',
          'ic': 'Inorganic Carbon (mg/L)'}

In [3]:
for trait in TITLES:
    print(f'Making truepreds for: {trait} ...')
    make_truepreds_for_trait(trait=trait,
                             title=TITLES[trait],
                             color=COLORS[1]) 

Making truepreds for: cfchl ...
Making truepreds for: cfpc ...
Making truepreds for: tss ...
Making truepreds for: npoc ...
Making truepreds for: no23 ...
Making truepreds for: secchi ...
Making truepreds for: tn ...
Making truepreds for: ic ...


In [4]:
for trait in TITLES:
    print(f'Making ndir2hists for: {trait} ...')
    make_ndi_r2_histogram_for_trait(trait=trait,
                                    title=TITLES[trait],
                                    color=COLORS[1]) 

Making ndir2hists for: cfchl ...
Making ndir2hists for: cfpc ...
Making ndir2hists for: tss ...
Making ndir2hists for: npoc ...
Making ndir2hists for: no23 ...
Making ndir2hists for: secchi ...
Making ndir2hists for: tn ...
Making ndir2hists for: ic ...


In [5]:
! tar -czf agu.tar.gz io/agu/